### Making a request
- access claude using the create function (`client.message.create()`), with 3 parameters
    - model: name of the claude model
    - max_tokens: the safety length of the response length ()
    - messages: conversations getting sent to claude

Messages Parameter:
- User message: context written by humans that gets sent to claude (a user or developer wants to be answered by claude)
- Assistant Messages: responses that a claude model has generated in response to the user message

In [17]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [19]:
# making a request
message = client.messages.create( 
    model=model, # name of the model
    max_tokens=250, # the maximum number of tokens that claude is allowed to generate, so if claude tries to generate more then 250 tokens, the generation will automatically stop, it wont reach for the number of max tokens, instead writing what it thinks is enough, stopping once the max is reached
    messages=[ # inputs to the model
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [20]:
print(message.content[0].text) # type: ignore

Quantum computing is a type of computation that uses quantum mechanical phenomena, such as superposition and entanglement, to process information in ways that can solve certain complex problems much faster than classical computers.


### Multi-Turn Conversations
- Anthropic API and claude do not store any messages or responses
- this means that in order to have a conversation or flow of messages (building on previous messages, like asking claude for another sentence on how quantum computing works) with claude we have to:
    - manually maintain a list of message in the code
    - and then provide the list with each follow up request

In [ ]:
message = client.messages.create( 
    model=model, 
    max_tokens=250,
    messages=[ 
        {
            "role": "user",
            "content": "Write another sentence" # asking a follow up questions about quantum computing to prove that claude does not save any information
        }
    ]
)

message.content[0].text # type: ignore
# this proves that claude does not store any information, since it is asking for subject of the sentence
# this second message(write another sentence) is the only context that claude is able to see, so the assistant does the best it can

"Could you provide more context? I'd be happy to write another sentence, but I'll need to know:\n\n- **What topic** or subject?\n- **What style** (formal, casual, creative, etc.)?\n- **What came before** it (if it's a continuation)?\n\nJust share some details and I'll help! 😊"

- The solution to this is after claude's response, add it back to the message list as an assistant message, then the new followup question (Write another sentence) is added as another user message, then this whole conversation history is sent to claude
- This simulates real conversation, where the users response then claude's response is saved one after the other with users message always coming first

- This way claude sees the entire history, both questions and responses

In [ ]:
# helper functions
# messages: the combined history of both users and claude's responses
# user_message: history of only the users questions
# assistant_message: history of only the assistant's responses

def add_user_message(messages, text): # list of messages, and the new text, question that was just asked from the user
    user_message = {"role": "user", "content": text} # role is always user, while content(new questions getting asked) will change 
    messages.append(user_message) # adding the new variable to the list of messages

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages): # passes in the history of the chat(user and claude both included)
    message = client.messages.create(
        model=model,
        max_tokens=250,
        messages=messages, # instead of defining questions and messages, the full history of the chat is given
    )
    return message.content[0].text # type: ignore
    # returning the final new message in response to the entire history of the conversation

In [23]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")
# the set would look like: {'role': 'user', 'content': 'Define quantum computing in one sentence'}

# Get Claude's response
answer = chat(messages) # this only includes the initial message

# Add Claude's response to the conversation history
add_assistant_message(messages, answer) # now messages includes claude's response, by adding claude's answer

# Add a follow-up question
add_user_message(messages, "Write another sentence") 
# messages has the first question, the response from claude and the continuation of: write another sentence

# Get the follow-up response with full context
final_answer = chat(messages) # since this is the final response, chat is called so we can view what claude said
final_answer

'Unlike classical computers, which store and process information as binary bits (0s and 1s), quantum computers use **qubits** that can exist in multiple states simultaneously, enabling them to perform many calculations at once and tackle problems in fields like cryptography, drug discovery, and optimization that would be practically impossible for traditional machines.'

### Chat Bot Exercise

In [ ]:
messages = []

while userInput != "End Conversation":
    userInput = input()
    print(userInput)
    
    add_user_message(messages, userInput)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    
    print(answer)

what is quantum computing
# Quantum Computing

## Basic Concept
Quantum computing is a type of computation that uses **quantum mechanical phenomena** to process information, fundamentally different from classical computing.

---

## Key Differences from Classical Computing

| Classical Computing | Quantum Computing |
|---------------------|-------------------|
| Uses **bits** (0 or 1) | Uses **qubits** |
| Deterministic | Probabilistic |
| Linear processing | Parallel processing |

---

## Core Principles

### 🔹 Superposition
- A qubit can exist as **0 AND 1 simultaneously**
- Like a coin spinning in the air (both heads and tails)

### 🔹 Entanglement
- Qubits can be **linked together**
- Changing one instantly affects another, regardless of distance

### 🔹 Interference
- Used to **amplify correct answers** and cancel wrong ones

---

## What Can It Do?
- Break/improve encryption
- Drug discovery & molecular simulation
- Optimization problems
- AI and machine learning
- Financial modeli